# PATSTAT inventor countries — `countries`, `n_countries`, first-inventor country

One row per application in the universe with at least one inventor, keyed on `appln_id`. The twin of
`PatentView/notebook/patent_inventor_country.ipynb` (and of the paper side's `paper_author_country`).

## Raw data
```
raw/tls207  pers_appln   person_id, appln_id, invt_seq_nr / applt_seq_nr
raw/tls206  person       person_id, person_ctry_code (ISO2; two blanks when unknown; ~120 junk codes such as
                         'N ', '75', '@@', 'N;' are read as unknown -- only [A-Z]{2} counts as a country)
PATSTAT/output/patstat_metadata.parquet   universe; inventor_ctry_list / applicant_ctry_list, the cross-check
PATSTAT/output/patstat_inventor.parquet   n_inventors, the cross-check
```
An inventor's country is `person_ctry_code` of **that person record** — the address as filed on that
application, the counterpart of PatentView's location attached to the inventor on that patent. One record
has one country, so `country_inventor_counts` sums exactly to `n_located` (unlike the paper files, where an
author can carry two). Namibia is `NA`; the raw parquet keeps it as a string.

## Output — `PATSTAT/output/patstat_inventor_country.parquet`
| column | |
|---|---|
| `appln_id` | application with ≥ 1 inventor row |
| `n_inventors` | inventors on the application (identical to `patstat_inventor.n_inventors`) |
| `n_located` | inventors with a country |
| `countries` | distinct ISO2 codes over located inventors, **sorted**, `;`-joined; null if none |
| `n_countries` | `len(countries.split(';'))`, 0 when null |
| `country_inventor_counts` | inventors per country, `US:3;JP:1`, count desc then code |
| `first_inventor_country` | country of the lowest-sequence inventor; null if *that* inventor is unlocated (not promoted) |
| `is_international` | `n_countries > 1` |
| `applicant_countries` | the same construction over the application's applicants (`applt_seq_nr > 0`) — PatentView's `assignee_countries` |

## One slot per person — and the unnamed placeholder
tls207 is **not** unique on `(person_id, appln_id)`: the same person can be listed at several sequence
numbers of one application. Two different things cause it, and they are treated differently:
- a **named** person listed twice (e.g. `KORITZ,DIETER,DD` at inventor positions 1 and 7) is one inventor —
  kept once, at the lowest sequence (PatentView's rule);
- an **unnamed** person record (`tls206.person_name` empty: person_id 263 `-NOT AVAILABLE-` and 83 others) is a
  placeholder that stands for several unrelated people — on 14,030 applications it fills more than one
  inventor slot — so each of its slots counts as a separate inventor and its id can repeat in the list.

`patstat_metadata.inventor_list`, this file and `patstat_inventor_country.n_inventors` all apply this rule.

Built in spillable steps (`count` / `max` groupings, then `string_agg` only over the small
(application, country) table): DuckDB cannot spill `list()` / `string_agg` states, and the inventor table has
~250 M rows.

> **Clock switch.** This text describes the default filing clock (`output/`). With `NB_PS_CLOCK=grant` the same notebook runs on the grant clock, PatentView's: years and ages are grant years (first publication with `publn_first_grant = 'Y'`), only applications with a grant year are in the universe (both ends of every edge), year / age columns are named `grant_year` / `yrs_since_grant`, and the outputs go to `PATSTAT/output_grant/` (executed copy: `notebook/grant/`).

> **Family unit.** With `NB_PS_UNIT=family` the same notebook runs with the DOCDB family as the unit (`docdb_family_id`), dated by the family's earliest priority year (`priority_year`, `yrs_since_priority`): citations are distinct family-to-family pairs with within-family citations dropped, classification comes from the earliest member that has one, inventors and their countries from one representative member (the one with the most located inventors), and the outputs go to `PATSTAT/output_family/` (executed copy: `notebook/family/`).


In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP  = ps.out('patstat_inventor_country.parquet')
META_FP = ps.out('patstat_metadata.parquet')
INV_FP  = ps.out('patstat_inventor.parquet')
ps.preflight('patstat_inventor_country')
con = ps.connect()

dump   : /project/jevans/PATSTAT/unzipped_data   (65 zip parts, 11 tables registered)
raw    : /project/jevans/Dawoon/Science of Science/PATSTAT/raw
output : /project/jevans/Dawoon/Science of Science/PATSTAT/output

  patstat_inventor_country      OK


## 1. Build

In [2]:
%%time
t0 = time.time()
# (appln_id, doc): the applications whose inventors are read and the unit they count for -- each application
# itself, or (NB_PS_UNIT=family) the representative member inv_appln_id chosen in patstat_metadata
DOCS = (f"SELECT inv_appln_id AS appln_id, docdb_family_id AS doc FROM read_parquet('{META_FP}') WHERE inv_appln_id IS NOT NULL"
        if ps.UNIT == 'family' else f"SELECT appln_id, appln_id AS doc FROM read_parquet('{META_FP}')")
# one row per (application, person, role slot); LEFT JOIN tls206 so a person without a record still counts as unlocated
con.execute(f"""CREATE OR REPLACE TABLE pr AS
  SELECT dm.doc, pa.person_id, pa.invt_seq_nr, pa.applt_seq_nr, CASE WHEN regexp_full_match(p.person_ctry_code, '[A-Z]{{2}}') THEN p.person_ctry_code END AS country,
         coalesce(trim(p.person_name) = '', false) AS unnamed
  FROM {ps.raw('tls207')} pa JOIN ({DOCS}) dm USING (appln_id) LEFT JOIN {ps.raw('tls206')} p USING (person_id)
  WHERE (pa.invt_seq_nr > 0 OR pa.applt_seq_nr > 0)""")
# inventor slots under the rule above: a named person once per application, an unnamed record once per slot
con.execute("""CREATE OR REPLACE TABLE inv AS
  WITH d AS (SELECT doc, person_id, min(invt_seq_nr) AS seq, any_value(country) AS country FROM pr
             WHERE invt_seq_nr > 0 GROUP BY doc, person_id, CASE WHEN unnamed THEN invt_seq_nr END)
  SELECT doc, person_id, country, row_number() OVER (PARTITION BY doc ORDER BY seq, person_id) AS pos FROM d""")
print(f'[{time.time()-t0:.0f}s] ' + '  '.join(f'{k} {v:,}' for k, v in zip(
    ['person-role rows', 'inventor slots', 'located inventors'],
    con.execute("SELECT (SELECT count(*) FROM pr), count(*), count(country) FROM inv").fetchone())))
con.execute(f"""COPY (
  WITH s AS (SELECT doc, count(*) AS n_inventors, count(country) AS n_located,
                    max(country) FILTER (WHERE pos = 1) AS first_inventor_country
             FROM inv GROUP BY 1),
       pc AS (SELECT doc, country, count(*) AS n FROM inv WHERE country IS NOT NULL GROUP BY 1, 2),
       c AS (SELECT doc, string_agg(country, ';' ORDER BY country) AS countries, count(*) AS n_countries,
                    string_agg(country || ':' || n, ';' ORDER BY n DESC, country) AS country_inventor_counts
             FROM pc GROUP BY 1),
       ac AS (SELECT DISTINCT doc, country FROM pr WHERE applt_seq_nr > 0 AND country IS NOT NULL),
       a AS (SELECT doc, string_agg(country, ';' ORDER BY country) AS applicant_countries FROM ac GROUP BY 1)
  SELECT s.doc AS {ps.KEY},
         s.n_inventors::INTEGER                 AS n_inventors,
         s.n_located::INTEGER                   AS n_located,
         c.countries,
         coalesce(c.n_countries, 0)::INTEGER    AS n_countries,
         c.country_inventor_counts,
         s.first_inventor_country,
         coalesce(c.n_countries, 0) > 1         AS is_international,
         a.applicant_countries
  FROM s LEFT JOIN c USING (doc) LEFT JOIN a USING (doc)
  ORDER BY s.doc
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
n = pq.ParquetFile(OUT_FP).metadata.num_rows
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e6:.1f} MB, {n:,} {ps.UNITS}) in {time.time()-t0:.0f}s')

[28s] person-role rows 274,783,517  inventor slots 195,446,903  located inventors 90,410,362
WROTE /project/jevans/Dawoon/Science of Science/PATSTAT/output/patstat_inventor_country.parquet  (235.1 MB, 75,617,406 applications) in 117s


## 2. Verification

1. `appln_id` unique; `n_located ≤ n_inventors`; `n_countries == len(countries.split(';'))`; sorted, no duplicates.
2. `country_inventor_counts` names exactly the codes in `countries` and sums to `n_located`.
3. `first_inventor_country` ∈ `countries`; `is_international` agrees with `n_countries`.
4. Against `patstat_inventor.n_inventors` (same applications, same counts) and against
   `patstat_metadata.inventor_ctry_list` / `applicant_ctry_list` (same strings — built there by a single
   `string_agg(DISTINCT …)` over the same rows).
5. Coverage by filing decade and by office, and the most frequent inventor countries.

In [3]:
%%time
chk = con.execute(f"""
WITH x AS (SELECT *, CASE WHEN countries IS NULL THEN [] ELSE str_split(countries, ';') END AS parts,
                     CASE WHEN country_inventor_counts IS NULL THEN []
                          ELSE list_transform(str_split(country_inventor_counts, ';'), z -> str_split(z, ':')) END AS cc
           FROM read_parquet('{OUT_FP}'))
SELECT count(*)                                                                  AS applications,
       count(*) - count(DISTINCT {ps.KEY})                                       AS dup_appln_id,
       sum(n_inventors)                                                          AS inventor_slots,
       sum(n_located)                                                            AS located_slots,
       count(*) FILTER (WHERE n_located > n_inventors)                           AS located_gt_inventors,
       count(*) FILTER (WHERE n_countries <> len(parts))                         AS n_countries_mismatch,
       count(*) FILTER (WHERE len(parts) <> len(list_distinct(parts)))           AS dup_in_countries,
       count(*) FILTER (WHERE parts <> list_sort(parts))                         AS unsorted_countries,
       count(*) FILTER (WHERE list_sum(list_transform(cc, z -> z[2]::INTEGER)) IS DISTINCT FROM nullif(n_located, 0)) AS counts_sum_mismatch,
       count(*) FILTER (WHERE list_sort(list_transform(cc, z -> z[1])) <> parts)  AS counts_codes_mismatch,
       count(*) FILTER (WHERE first_inventor_country IS NOT NULL AND NOT list_contains(parts, first_inventor_country)) AS first_not_in_list,
       count(*) FILTER (WHERE is_international <> (n_countries > 1))             AS intl_mismatch,
       count(*) FILTER (WHERE countries IS NOT NULL)                             AS with_country,
       count(*) FILTER (WHERE is_international)                                  AS international,
       count(*) FILTER (WHERE applicant_countries IS NOT NULL)                   AS with_applicant_country
FROM x""").fetchdf().iloc[0]
for k, v in chk.items():
    print(f'  {k:<24} {int(v):>14,}')
HARD = ['dup_appln_id', 'located_gt_inventors', 'n_countries_mismatch', 'dup_in_countries', 'unsorted_countries',
        'counts_sum_mismatch', 'counts_codes_mismatch', 'first_not_in_list', 'intl_mismatch']
ok = all(chk[k] == 0 for k in HARD)
print(f"\n1-3. {'ALL PASS' if ok else 'FAILED'}   located: {chk.with_country/chk.applications*100:.1f}% of {ps.UNITS}, "
      f"{chk.located_slots/chk.inventor_slots*100:.1f}% of inventor slots; international {chk.international/chk.applications*100:.1f}%")

x = con.execute(f"""
WITH n AS (SELECT {ps.KEY}, n_inventors, countries, applicant_countries FROM read_parquet('{OUT_FP}')),
     i AS (SELECT {ps.KEY}, n_inventors FROM read_parquet('{INV_FP}')),
     m AS (SELECT {ps.KEY}, inventor_ctry_list, applicant_ctry_list FROM read_parquet('{META_FP}'))
SELECT (SELECT count(*) FROM n JOIN i USING ({ps.KEY}))                                         AS shared_inventor,
       (SELECT count(*) FROM n JOIN i USING ({ps.KEY}) WHERE n.n_inventors <> i.n_inventors)    AS n_inventors_mismatch,
       (SELECT count(*) FROM n ANTI JOIN i USING ({ps.KEY})) + (SELECT count(*) FROM i ANTI JOIN n USING ({ps.KEY})) AS one_sided,
       (SELECT count(*) FROM n JOIN m USING ({ps.KEY}) WHERE n.countries IS DISTINCT FROM m.inventor_ctry_list)  AS inventor_ctry_mismatch,
       (SELECT count(*) FROM n JOIN m USING ({ps.KEY}) WHERE n.applicant_countries IS DISTINCT FROM m.applicant_ctry_list) AS applicant_ctry_mismatch
""").fetchdf().iloc[0]
print('\n4.   ' + '  '.join(f'{k} {int(v):,}' for k, v in x.items()))
ok4 = x.n_inventors_mismatch == 0 and x.one_sided == 0 and x.inventor_ctry_mismatch == 0 and x.applicant_ctry_mismatch == 0
print(f"     vs patstat_inventor and patstat_metadata: {'OK' if ok4 else 'FAILED'}")

print('\n5.   coverage by filing decade')
display(con.execute(f"""SELECT (m.filing_year // 10) * 10 AS decade, count(*) AS applications,
       round(100.0 * count(s.countries) / count(*), 1) AS pct_located,
       round(100.0 * count(*) FILTER (WHERE s.is_international) / count(*), 2) AS pct_international,
       round(avg(s.n_inventors), 2) AS mean_inventors
FROM read_parquet('{OUT_FP}') s JOIN read_parquet('{META_FP}') m USING ({ps.KEY}) GROUP BY 1 ORDER BY 1""").fetchdf())
print('     by filing office (top 12)')
display(con.execute(f"""SELECT m.appln_auth, count(*) AS applications, round(100.0 * count(s.countries) / count(*), 1) AS pct_located,
       round(100.0 * count(*) FILTER (WHERE s.is_international) / count(*), 2) AS pct_international
FROM read_parquet('{OUT_FP}') s JOIN read_parquet('{META_FP}') m USING ({ps.KEY}) GROUP BY 1 ORDER BY 2 DESC LIMIT 12""").fetchdf())
print('     most frequent inventor countries (applications with >= 1 inventor there)')
display(con.execute(f"""SELECT c AS country, count(*) AS applications,
       round(100.0 * count(*) / (SELECT count(*) FROM read_parquet('{OUT_FP}')), 2) AS pct
FROM read_parquet('{OUT_FP}'), unnest(str_split(countries, ';')) AS u(c)
WHERE countries IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 15""").fetchdf())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') WHERE is_international ORDER BY n_countries DESC LIMIT 5").fetchdf())
con.close()
assert ok and ok4

  applications                 75,617,406
  dup_appln_id                          0
  inventor_slots              195,446,903
  located_slots                90,410,362
  located_gt_inventors                  0
  n_countries_mismatch                  0
  dup_in_countries                      0
  unsorted_countries                    0
  counts_sum_mismatch                   0
  counts_codes_mismatch                 0
  first_not_in_list                     0
  intl_mismatch                         0
  with_country                 35,618,480
  international                 2,059,609
  with_applicant_country       38,846,662

1-3. ALL PASS   located: 47.1% of applications, 46.3% of inventor slots; international 2.7%

4.   shared_inventor 75,617,406  n_inventors_mismatch 0  one_sided 0  inventor_ctry_mismatch 0  applicant_ctry_mismatch 0
     vs patstat_inventor and patstat_metadata: OK

5.   coverage by filing decade


,decade,applications,pct_located,pct_international,mean_inventors
0,1900,674364,86.0,0.17,1.14
1,1910,691920,79.8,0.17,1.13
2,1920,698080,11.2,0.04,1.12
3,1930,697353,11.6,0.04,1.16
4,1940,627594,12.1,0.02,1.19
5,1950,1018012,11.4,0.09,1.34
6,1960,1551627,18.1,0.15,1.54
7,1970,3377906,19.0,0.19,2.00
8,1980,7057108,36.9,0.50,2.21
9,1990,9918257,53.6,2.26,2.29


     by filing office (top 12)


,appln_auth,applications,pct_located,pct_international
0,CN,15836153,7.5,0.00
1,US,13602267,81.4,4.80
2,JP,13177197,0.2,0.01
3,WO,4638387,93.2,6.00
4,DE,4277920,64.7,2.66
5,EP,4197027,99.9,6.96
6,KR,3785477,74.6,3.35
7,CA,1742478,80.6,5.50
8,AU,1441177,1.0,0.08
9,FR,1259273,31.3,0.67


     most frequent inventor countries (applications with >= 1 inventor there)


,country,applications,pct
0,US,11084218,14.66
1,DE,4715807,6.24
2,JP,4698746,6.21
3,KR,3246080,4.29
4,CN,1938451,2.56
5,FR,1672167,2.21
6,GB,1440476,1.90
7,TW,857117,1.13
8,CH,846572,1.12
9,CA,730313,0.97


,appln_id,n_inventors,n_located,countries,n_countries,country_inventor_counts,first_inventor_country,is_international,applicant_countries
0,527197344,130,130,AT;BR;CN;DE;ES;FI;FR;GR;IN;IR;KR;NO;PK;RS;SE;T...,19,SE:83;CN:12;DE:7;US:4;FI:3;IN:3;TR:3;IR:2;KR:2...,DE,True,SE
1,510798336,130,130,AT;BR;CN;DE;ES;FI;FR;GR;IN;IR;KR;NO;PK;RS;SE;T...,19,SE:85;CN:12;DE:7;US:4;FI:3;IN:2;IR:2;KR:2;PK:2...,CN,True,SE
2,544283169,65,65,AT;CA;CN;DE;ES;FI;GR;HU;IN;IR;IT;NL;PS;PT;RU;S...,19,SE:28;HU:9;DE:5;FI:4;US:3;IN:2;RU:2;AT:1;CA:1;...,DE,True,SE
3,562115999,65,65,AT;CA;CN;DE;ES;FI;GR;HU;IN;IR;IT;NL;PS;PT;RU;S...,19,SE:28;HU:9;DE:5;FI:4;US:3;IN:2;RU:2;AT:1;CA:1;...,DE,True,SE
4,511593583,130,130,AU;BR;CN;DE;ES;FI;FR;GR;IN;IR;KR;NO;PK;SE;TN;T...,18,SE:84;CN:12;DE:7;US:4;FI:3;IN:3;TR:3;IR:2;KR:2...,AU,True,SE
